In [9]:
# ==========================
# Task 3: Hypothesis Testing (Cleaned Data)
# ==========================

import os
import pandas as pd
import numpy as np
from scipy import stats

# Step 1: Load cleaned/preprocessed data
data_path = "../data/processed/cleaned_data.csv"
df.to_csv("../data/processed/cleaned_data.csv", index=False)

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Cleaned CSV not found at {data_path}")

df = pd.read_csv(data_path)

# Step 2: Ensure key metrics exist
if 'claim_occurred' not in df.columns:
    df['claim_occurred'] = np.where(df['totalclaims'] > 0, 1, 0)

if 'claim_severity' not in df.columns:
    df['claim_severity'] = df.apply(lambda x: x['totalclaims'] if x['claim_occurred'] == 1 else np.nan, axis=1)

if 'margin' not in df.columns:
    df['margin'] = df['totalpremium'] - df['totalclaims']

# Helper function for t-test and result printing
def perform_ttest(group_a, group_b, hypothesis_name):
    t_stat, p_val = stats.ttest_ind(group_a, group_b, nan_policy='omit')
    result = "Reject H0" if p_val < 0.05 else "Fail to reject H0"
    print(f"{hypothesis_name}: t-stat={t_stat:.3f}, p-value={p_val:.5f}, Result={result}")
    return t_stat, p_val, result

# Step 3: H1 - Risk differences across provinces
provinces = df['province'].dropna().unique()
if len(provinces) >= 2:
    group_a = df[df['province'] == provinces[0]]['claim_occurred']
    group_b = df[df['province'] == provinces[1]]['claim_occurred']
    h1_stat, h1_pval, h1_result = perform_ttest(group_a, group_b, "H1 Province")
else:
    print("Not enough provinces to test H1")

# Step 4: H2 - Risk differences across zip codes
zip_codes = df['postalcode'].dropna().unique()
if len(zip_codes) >= 2:
    group_a = df[df['postalcode'] == zip_codes[0]]['claim_occurred']
    group_b = df[df['postalcode'] == zip_codes[1]]['claim_occurred']
    h2_stat, h2_pval, h2_result = perform_ttest(group_a, group_b, "H2 Zip Code (Risk)")
else:
    print("Not enough zip codes to test H2")

# Step 5: H3 - Margin differences across zip codes
if len(zip_codes) >= 2:
    group_a = df[df['postalcode'] == zip_codes[0]]['margin']
    group_b = df[df['postalcode'] == zip_codes[1]]['margin']
    h3_stat, h3_pval, h3_result = perform_ttest(group_a, group_b, "H3 Margin by Zip Code")
else:
    print("Not enough zip codes to test H3")

# Step 6: H4 - Gender differences
genders = df['gender'].dropna().unique()
if len(genders) >= 2:
    group_a = df[df['gender'] == genders[0]]['claim_occurred']
    group_b = df[df['gender'] == genders[1]]['claim_occurred']
    h4_stat, h4_pval, h4_result = perform_ttest(group_a, group_b, "H4 Gender")
else:
    print("Not enough gender categories to test H4")

# Step 7: Summary table
summary_df = pd.DataFrame({
    'Hypothesis': ["H1 Province", "H2 Zip Code (Risk)", "H3 Margin by Zip Code", "H4 Gender"],
    't_stat': [h1_stat, h2_stat, h3_stat, h4_stat],
    'p_value': [h1_pval, h2_pval, h3_pval, h4_pval],
    'Result': [h1_result, h2_result, h3_result, h4_result]
})

print("\n===== Summary =====")
print(summary_df)


C:\Users\user\AppData\Local\Temp\ipykernel_17348\3023142910.py:17: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


H1 Province: t-stat=3.119, p-value=0.00181, Result=Reject H0
H2 Zip Code (Risk): t-stat=nan, p-value=nan, Result=Fail to reject H0
H3 Margin by Zip Code: t-stat=-0.486, p-value=0.62687, Result=Fail to reject H0
H4 Gender: t-stat=2.440, p-value=0.01468, Result=Reject H0

===== Summary =====
              Hypothesis    t_stat   p_value             Result
0            H1 Province  3.119195  0.001814          Reject H0
1     H2 Zip Code (Risk)       NaN       NaN  Fail to reject H0
2  H3 Margin by Zip Code -0.486346  0.626874  Fail to reject H0
3              H4 Gender  2.440278  0.014676          Reject H0


In [10]:
# ==========================
# Task 3: Comprehensive Hypothesis Testing
# ==========================

import os
import pandas as pd
import numpy as np
from scipy import stats

# --------------------------
# Step 1: Load cleaned data
# --------------------------
data_path = "../data/processed/cleaned_data.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Cleaned CSV not found at {data_path}")

df = pd.read_csv(data_path)

# Ensure necessary metrics exist
df['claim_occurred'] = np.where(df['totalclaims'] > 0, 1, 0)
df['claim_severity'] = np.where(df['claim_occurred'] == 1, df['totalclaims'], np.nan)
df['margin'] = df['totalpremium'] - df['totalclaims']

# --------------------------
# Step 2: Helper function for t-tests
# --------------------------
def ttest_metric(df, feature, value_a, value_b, metric):
    """Performs independent t-test for a given metric between two groups."""
    group_a = df[df[feature] == value_a][metric].dropna()
    group_b = df[df[feature] == value_b][metric].dropna()
    
    # Check sample sizes
    if len(group_a) < 10 or len(group_b) < 10:
        return np.nan, np.nan, "Insufficient data"
    
    t_stat, p_val = stats.ttest_ind(group_a, group_b, nan_policy='omit')
    result = "Reject H0" if p_val < 0.05 else "Fail to reject H0"
    return t_stat, p_val, result

# --------------------------
# Step 3: Define comparisons
# --------------------------
comparisons = {
    "H1 Province": {"feature": "province", "metric": "claim_occurred"},
    "H2 Zip Code (Risk)": {"feature": "postalcode", "metric": "claim_occurred"},
    "H3 Margin by Zip Code": {"feature": "postalcode", "metric": "margin"},
    "H4 Gender": {"feature": "gender", "metric": "claim_occurred"},
    "H5 Claim Severity by Province": {"feature": "province", "metric": "claim_severity"},
}

summary = []

# --------------------------
# Step 4: Execute tests
# --------------------------
for hypothesis, params in comparisons.items():
    feature = params["feature"]
    metric = params["metric"]
    unique_values = df[feature].dropna().unique()
    
    if len(unique_values) < 2:
        summary.append([hypothesis, np.nan, np.nan, "Insufficient categories", "Not enough data"])
        continue
    
    # Compare first two unique categories (can enhance later with proper segmentation)
    t_stat, p_val, result = ttest_metric(df, feature, unique_values[0], unique_values[1], metric)
    
    # --------------------------
    # Step 5: Business Recommendation
    # --------------------------
    if result == "Reject H0":
        mean_a = df[df[feature]==unique_values[0]][metric].mean()
        mean_b = df[df[feature]==unique_values[1]][metric].mean()
        perc_diff = (mean_a - mean_b) / mean_b * 100 if mean_b != 0 else np.nan
        recommendation = f"{unique_values[0]} has {perc_diff:.2f}% higher {metric.replace('_',' ')} than {unique_values[1]}. Consider adjusting premiums or segmentation accordingly."
    else:
        recommendation = "No significant difference detected. No immediate action recommended."
    
    summary.append([hypothesis, t_stat, p_val, result, recommendation])

# --------------------------
# Step 6: Summary Table
# --------------------------
summary_df = pd.DataFrame(summary, columns=["Hypothesis", "t_stat", "p_value", "Result", "Business Recommendation"])
pd.set_option("display.max_colwidth", None)
print(summary_df)


C:\Users\user\AppData\Local\Temp\ipykernel_17348\1591271306.py:17: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


                      Hypothesis    t_stat   p_value             Result  \
0                    H1 Province  3.119195  0.001814          Reject H0   
1             H2 Zip Code (Risk)       NaN       NaN  Fail to reject H0   
2          H3 Margin by Zip Code -0.486346  0.626874  Fail to reject H0   
3                      H4 Gender  2.440278  0.014676          Reject H0   
4  H5 Claim Severity by Province -3.674482  0.000245          Reject H0   

                                                                                                  Business Recommendation  
0   Gauteng has 17.98% higher claim occurred than KwaZulu-Natal. Consider adjusting premiums or segmentation accordingly.  
1                                                    No significant difference detected. No immediate action recommended.  
2                                                    No significant difference detected. No immediate action recommended.  
3      Not specified has 29.05% higher claim occurred

In [1]:
# ==========================
# Module: Uncovered Task 3 Analysis
# ==========================
import os
import pandas as pd
import numpy as np
from scipy import stats

# Paths
data_path = "../data/processed/cleaned_data.csv"
results_path = "../data/processed/task3_additional_results.csv"

# Load Data
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Cleaned CSV not found at {data_path}")

df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Ensure metrics exist
df['claim_occurred'] = np.where(df['totalclaims'] > 0, 1, 0)
df['claim_severity'] = df.apply(lambda x: x['totalclaims'] if x['claim_occurred'] == 1 else np.nan, axis=1)
df['margin'] = df['totalpremium'] - df['totalclaims']

# ==========================
# Segmentation Validation
# ==========================
def check_equivalence(df, feature):
    categories = df[feature].dropna().unique()
    if len(categories) < 2:
        return None
    group_a = df[df[feature] == categories[0]]
    group_b = df[df[feature] == categories[1]]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    equivalence_report = {}
    for col in numeric_cols:
        t_stat, p_val = stats.ttest_ind(group_a[col], group_b[col], nan_policy='omit')
        equivalence_report[col] = p_val >= 0.05
    return equivalence_report

# Example: Provinces equivalence
equiv_province = check_equivalence(df, 'province')
print("Segmentation equivalence by Province:", equiv_province)

# ==========================
# Claim Severity Tests (Uncovered)
# ==========================
results = []

def run_ttest(group_a, group_b, name):
    t_stat, p_val = stats.ttest_ind(group_a.dropna(), group_b.dropna(), nan_policy='omit')
    result = "Reject H0" if p_val < 0.05 else "Fail to reject H0"
    return t_stat, p_val, result

# H5: Claim Severity by Province
provinces = df['province'].dropna().unique()
if len(provinces) >= 2:
    group_a = df[df['province'] == provinces[0]]['claim_severity']
    group_b = df[df['province'] == provinces[1]]['claim_severity']
    t_stat, p_val, result = run_ttest(group_a, group_b, "H5 Claim Severity by Province")
    diff = group_a.mean() - group_b.mean()
    recommendation = f"{provinces[0]} has {diff:.2f} higher claim severity than {provinces[1]}. Adjust premiums if needed."
    results.append(["H5 Claim Severity by Province", t_stat, p_val, result, recommendation])

# H6: Claim Severity by Gender
genders = df['gender'].dropna().unique()
if len(genders) >= 2:
    group_a = df[df['gender'] == genders[0]]['claim_severity']
    group_b = df[df['gender'] == genders[1]]['claim_severity']
    t_stat, p_val, result = run_ttest(group_a, group_b, "H6 Claim Severity by Gender")
    diff = group_a.mean() - group_b.mean()
    recommendation = f"{genders[0]} has {diff:.2f} higher claim severity than {genders[1]}. Consider adjusting segmentation."
    results.append(["H6 Claim Severity by Gender", t_stat, p_val, result, recommendation])

# Optional: Granular tests (vehicle type, legal type)
vehicle_types = df['vehicletype'].dropna().unique()
if len(vehicle_types) >= 2:
    group_a = df[df['vehicletype'] == vehicle_types[0]]['claim_severity']
    group_b = df[df['vehicletype'] == vehicle_types[1]]['claim_severity']
    t_stat, p_val, result = run_ttest(group_a, group_b, "H7 Claim Severity by Vehicle Type")
    diff = group_a.mean() - group_b.mean()
    recommendation = f"{vehicle_types[0]} has {diff:.2f} higher claim severity than {vehicle_types[1]}."
    results.append(["H7 Claim Severity by Vehicle Type", t_stat, p_val, result, recommendation])

# Save results
results_df = pd.DataFrame(results, columns=["Hypothesis", "t_stat", "p_value", "Result", "Business_Recommendation"])
results_df.to_csv(results_path, index=False)
print("Additional uncovered hypothesis results saved to:", results_path)
print(results_df)


D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\venv\Lib\site-packages\scipy\_lib\deprecation.py:234: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)


Segmentation equivalence by Province: {'underwrittencoverid': np.False_, 'policyid': np.False_, 'postalcode': np.False_, 'mmcode': np.False_, 'registrationyear': np.False_, 'cylinders': np.False_, 'cubiccapacity': np.False_, 'kilowatts': np.False_, 'numberofdoors': np.True_, 'customvalueestimate': np.False_, 'numberofvehiclesinfleet': np.False_, 'suminsured': np.True_, 'calculatedpremiumperterm': np.False_, 'totalpremium': np.False_, 'totalclaims': np.True_, 'claim_occurred': np.False_, 'claim_severity': np.False_, 'margin': np.True_}
Additional uncovered hypothesis results saved to: ../data/processed/task3_additional_results.csv
                          Hypothesis    t_stat   p_value             Result  \
0      H5 Claim Severity by Province -3.674482  0.000245          Reject H0   
1        H6 Claim Severity by Gender  2.130206  0.033243          Reject H0   
2  H7 Claim Severity by Vehicle Type -1.005574  0.314709  Fail to reject H0   

                             Business_Recomme